# HumanizingSpeech — Voice-Cloned Expressive Narration (Colab GPU)

Run the prosody pipeline on a **GPU runtime** and render the final audio in **your own cloned voice** via XTTS-v2.

**Before you start:** `Runtime -> Change runtime type -> GPU` (T4 is fine).

Steps:
1. Check the GPU
2. Clone the repo + install dependencies
3. Provide a reference clip of your voice (record in-browser or upload)
4. Provide your story text
5. Generate expressive, voice-cloned audio and play it inline

## 1. Check GPU

In [ ]:
!nvidia-smi
import torch
print('CUDA available:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'No GPU! Runtime -> Change runtime type -> GPU, then re-run.'

## 2. Clone the repo + install dependencies

Paste a **fresh** GitHub token below (the old one in your git remote was exposed — rotate it). If you make the repo public you can leave the token blank.

In [ ]:
GITHUB_USER  = 'm25ai2101'  #@param {type:'string'}
GITHUB_TOKEN = ''           #@param {type:'string'}
REPO         = 'HumanisingSpeechForKids'  #@param {type:'string'}
BRANCH       = 'voice-cloning'  #@param {type:'string'}

import os
auth = f'{GITHUB_USER}:{GITHUB_TOKEN}@' if GITHUB_TOKEN else ''
url  = f'https://{auth}github.com/{GITHUB_USER}/{REPO}.git'
if not os.path.isdir(REPO):
    !git clone --branch $BRANCH $url
%cd $REPO
!git checkout $BRANCH
!git pull origin $BRANCH

In [ ]:
# XTTS/coqui-tts needs Python < 3.12. Colab's default may be newer; if the
# install fails, use Runtime with an older Python or a compatible image.
import sys
print('Python:', sys.version)

# System libs: ffmpeg (torchcodec audio I/O), libportaudio2 (sounddevice),
# espeak-ng (pyttsx3 fallback).
!apt-get -qq install -y ffmpeg libportaudio2 espeak-ng > /dev/null

# Install project deps. torch/torchaudio are already present on Colab (CUDA build).
!pip install -q -r requirements.txt
print('\nDependencies installed. If coqui-tts failed on Python 3.12+, switch the runtime Python version.')

## 3. Your reference voice clip

**Option A — record in the browser** (run the cell, allow mic, click Stop after ~10-20s of expressive reading).

**Option B — upload a WAV** you already recorded (run the upload cell instead).

In [ ]:
#@title Option A: record in-browser -> my_voice.wav
from IPython.display import Javascript, display
from google.colab import output
from base64 import b64decode
import io, scipy.io.wavfile as wavio, numpy as np, subprocess, os

RECORD_JS = '''
const sleep = ms => new Promise(r => setTimeout(r, ms));
async function record() {
  const div = document.createElement('div');
  const btn = document.createElement('button');
  btn.textContent = 'Stop recording';
  div.appendChild(btn); document.body.appendChild(div);
  const stream = await navigator.mediaDevices.getUserMedia({audio:true});
  const rec = new MediaRecorder(stream); const chunks = [];
  rec.ondataavailable = e => chunks.push(e.data);
  const stopped = new Promise(r => btn.onclick = r);
  rec.start(); await stopped; rec.stop();
  await new Promise(r => rec.onstop = r);
  stream.getTracks().forEach(t => t.stop()); div.remove();
  const blob = new Blob(chunks);
  const buf = await blob.arrayBuffer();
  let bin = ''; const bytes = new Uint8Array(buf);
  for (const b of bytes) bin += String.fromCharCode(b);
  return btoa(bin);
}
'''
display(Javascript(RECORD_JS))
print('Recording... click "Stop recording" when done.')
b64 = output.eval_js('record()')
with open('_ref_raw.webm', 'wb') as f:
    f.write(b64decode(b64))
# Convert the browser recording to a clean mono 24 kHz WAV via ffmpeg.
subprocess.run(['ffmpeg','-y','-i','_ref_raw.webm','-ac','1','-ar','24000','my_voice.wav'],
               check=True, capture_output=True)
sr, data = wavio.read('my_voice.wav')
print(f'Saved my_voice.wav ({len(data)/sr:.1f}s @ {sr} Hz)')
from IPython.display import Audio; Audio('my_voice.wav')

In [ ]:
#@title Option B: upload a WAV instead -> my_voice.wav
from google.colab import files
up = files.upload()
name = next(iter(up))
import subprocess
subprocess.run(['ffmpeg','-y','-i',name,'-ac','1','-ar','24000','my_voice.wav'],
               check=True, capture_output=True)
from IPython.display import Audio; print('Saved my_voice.wav'); Audio('my_voice.wav')

## 4. Your story text

In [ ]:
story = '''Once upon a time, in a village at the edge of a dark wood, there lived a curious little fox.

"I wonder what lies beyond those trees," she whispered, her heart pounding with excitement.

But deep in the shadows, something was watching. Something old. Something hungry.'''

with open('story.txt', 'w', encoding='utf-8') as f:
    f.write(story)
print('Wrote story.txt')

## 5. Generate voice-cloned expressive audio

First run downloads the XTTS-v2 model (~1.9 GB) and loads it on the GPU.

In [ ]:
# play=False on Colab (no speakers); we display the audio widget afterwards.
!python pipeline.py --text-file story.txt --model-prosody \
    --clone-ref my_voice.wav --save output.wav

In [ ]:
from IPython.display import Audio
Audio('output.wav')

In [ ]:
from google.colab import files
files.download('output.wav')